In [2]:
# ── Celda 1: Imports ─────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Celda 2: Carga ───────────────────────────────────────────────
df = pd.read_csv('/content/count_412_912.csv')
print(f"Filas cargadas: {len(df)}")
print(df.info())

# ── Celda 3: Conversión timestamp ────────────────────────────────
df['fecha'] = pd.to_datetime(df['received'], unit='s')
df = df.sort_values('fecha').reset_index(drop=True)
print(f"\nRango temporal: {df['fecha'].min()} → {df['fecha'].max()}")

# ── Celda 4: Limpieza ────────────────────────────────────────────
# Duplicados
n_duplicados = df.duplicated(subset=['received']).sum()
df = df.drop_duplicates(subset=['received'])
print(f"Duplicados eliminados: {n_duplicados}")

# Nulos en body.counting
n_nulos = df['body.counting'].isna().sum()
df = df.dropna(subset=['body.counting'])
print(f"Filas con body.counting nulo eliminadas: {n_nulos}")

# ── Celda 5: Resample a 15min ────────────────────────────────────
df_15min = df.resample('15min', on='fecha')['body.counting'].agg(
    lambda x: x.max() - x.min() if len(x) > 0 else 0
).to_frame()

print(f"\nFilas después del resample: {len(df_15min)}")
print(df_15min['body.counting'].describe())

# ── Celda 6: Guardar ─────────────────────────────────────────────
df_15min.to_csv('../content/datos_procesados.csv')
print("\nGuardado en data/datos_procesados.csv")

Filas cargadas: 1359
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1359 entries, 0 to 1358
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   best_id        1359 non-null   object 
 1   body.counting  1323 non-null   float64
 2   body.voltage   475 non-null    float64
 3   received       1359 non-null   float64
dtypes: float64(3), object(1)
memory usage: 42.6+ KB
None

Rango temporal: 2025-12-04 03:03:20.707506895 → 2025-12-08 17:14:02.265219927
Duplicados eliminados: 0
Filas con body.counting nulo eliminadas: 36

Filas después del resample: 441
count    441.000000
mean      19.625850
std       14.011948
min        0.000000
25%        9.000000
50%       17.000000
75%       26.000000
max      111.000000
Name: body.counting, dtype: float64

Guardado en data/datos_procesados.csv
